# harden-v0 analysis notebook base

This notebook is for analyzing harden-v0 output data.
Instructions:

In [38]:
import duckdb
import pandas as pd
from pathlib import Path
import glob
import os

# define paths
batches_dir = Path("/home/ivgeni/truthserum/harden-v0/outputs")

jobs_parquet_path = "anvil_jobs.parquet"
configs_parquet_path = "anvil_configs.parquet"
trials_parquet_path = "anvil_trials.parquet"
trajectories_parquet_path = "anvil_trajectories.parquet"
trial_summaries_parquet_path = "anvil_trial_summaries.parquet"
step_summaries_parquet_path = "anvil_step_summaries.parquet"
tasks_parquet_path = "anvil_tasks.parquet"
iterations_parquet_path = "anvil_iterations.parquet"
pool_commits_parquet_path = "anvil_pool_commits.parquet"

# Directory structure:
#   shared_harden-v0/
#     BATCH_DIR/                                ← batch level (e.g. batch_TIMESTAMP/ or
#       batch_summary.json                      ←   ad-hoc names like tb-non-hacked-gemini-3-flash/)
#       batch_config.json
#       TASK_NAME/
#         result.json                           ← task result (status, iterations, hardened_dir, ...)
#         task_config.json
#         hacker_task/  solver_task/  hardened/
#         jobs/
#           JOB_ID/                             ← e.g. solver_precheck_a0__TIMESTAMP__HASH
#             config.json
#             result.json
#             job.log
#             TRIAL_NAME/                       ← e.g. task-name__HASH
#               config.json
#               result.json
#               trial.log
#               agent/
#                 trajectory.json               ← steps, final_metrics
#                 episode-N/                    ← per-episode debug data
#               verifier/  artifacts/
#
# JOB_ID naming: {type}[_iter{N}][_a{N}]__{timestamp}__{hash}
#
# NOTE: trajectory.summary.json files are no longer emitted by current harden-v0
# output, so the summary-related views may be empty.

# Auto-detect batch directories: any immediate subdirectory of `batches_dir`
# that contains at least one `TASK/jobs/` folder is treated as a batch. This
# covers both `batch_TIMESTAMP/` and ad-hoc run dirs (e.g. `tb-non-hacked-*`).
_batch_dirs = sorted(
    p for p in batches_dir.iterdir()
    if p.is_dir() and any(p.glob("*/jobs"))
)
print(f"Detected {len(_batch_dirs)} batch directories: {[p.name for p in _batch_dirs]}")

# Build one glob pattern per batch dir, per relation. DuckDB's read_json_auto
# accepts a list of glob patterns and does file enumeration internally —
# this is far cheaper than passing 1000s of literal paths.
# NOTE: DuckDB errors if ANY pattern in the list matches zero files, so we
# pre-filter to only patterns with at least one match.
def _globs(suffix):
    out = []
    for b in _batch_dirs:
        pat = str(b / suffix)
        if glob.glob(pat):
            out.append(pat)
    return out

batches_globs = _globs("batch_summary.json")
tasks_globs = _globs("*/result.json")
jobs_globs = _globs("*/jobs/*/result.json")
configs_globs = _globs("*/jobs/*/config.json")
trials_globs = _globs("*/jobs/*/*/result.json")
trajectories_globs = _globs("*/jobs/*/*/agent/trajectory.json")
traj_summaries_globs = _globs("*/jobs/*/*/agent/trajectory.summary.json")

# For sanity-display of file counts (uses Python glob, just for printing):
def _count(patterns):
    n = 0
    for p in patterns:
        n += len(glob.glob(p))
    return n

print(
    f"  batches: {_count(batches_globs)} | tasks: {_count(tasks_globs)} | "
    f"jobs: {_count(jobs_globs)} | configs: {_count(configs_globs)} | "
    f"trials: {_count(trials_globs)} | trajectories: {_count(trajectories_globs)} | "
    f"traj_summaries: {_count(traj_summaries_globs)}"
)

def _sql_globs(patterns):
    """Render a Python list of glob patterns as a DuckDB SQL array literal."""
    return "[" + ", ".join("'" + p.replace("'", "''") + "'" for p in patterns) + "]"

# Connect to an in-memory database
con = duckdb.connect()

# Larger memory limit + temp dir spill — the dataset can be ~10k+ JSON files.
con.execute("SET memory_limit='12GB'")
con.execute("SET preserve_insertion_order=false")
con.execute("SET threads=4")
con.execute("SET temp_directory='/tmp/duckdb_spill'")

def job_id_parse_sql(job_id_expr):
    """SQL fragment: extract job_type, iteration, attempt from a job_id expression."""
    prefix = f"string_split({job_id_expr}, '__')[1]"
    return f"""
        regexp_replace({prefix}, '(_iter[0-9]+)?(_a[0-9]+)?$', '') as job_type,
        try_cast(regexp_extract({prefix}, '_iter([0-9]+)', 1) as INTEGER) as iteration,
        try_cast(regexp_extract({prefix}, '_a([0-9]+)$', 1) as INTEGER) as attempt"""

def convert_to_parquet(name, patterns, parquet_path, build_query):
    """Stream JSON -> Parquet. `patterns` is a list of glob patterns. `build_query(L)`
    returns a SELECT given the SQL array literal."""
    if not patterns or _count(patterns) == 0:
        print(f"No {name} files found.")
        return False
    print(f"Converting {name} files to {parquet_path}...")
    try:
        con.execute(f"COPY ({build_query(_sql_globs(patterns))}) TO '{parquet_path}' (FORMAT PARQUET)")
        con.execute(f"CREATE OR REPLACE VIEW raw_{name} AS SELECT * FROM '{parquet_path}'")
        print(f"Success! Created view 'raw_{name}' from {parquet_path}")
        return True
    except Exception as e:
        print(f"Error converting {name}: {e}")
        return False

# Tasks: BATCH_DIR/TASK_NAME/result.json  ([-3]=batch, [-2]=task)
convert_to_parquet("tasks", tasks_globs, tasks_parquet_path, lambda L: f"""
    SELECT
        string_split(filename, '/')[-3] as batch_id,
        string_split(filename, '/')[-2] as task_name,
        task_id, status,
        len(iterations) as num_iterations,
        hardened_dir, oracle, pool_enabled, kernelbench_mode
    FROM read_json_auto({L}, union_by_name=true, filename=true)
""")

# Iterations (unnested from task results)
convert_to_parquet("iterations", tasks_globs, iterations_parquet_path, lambda L: f"""
    WITH raw AS (
        SELECT
            string_split(filename, '/')[-3] as batch_id,
            string_split(filename, '/')[-2] as task_name,
            task_id,
            unnest(iterations) as iter
        FROM read_json_auto({L}, union_by_name=true, filename=true)
    )
    SELECT
        batch_id, task_name, task_id,
        iter.iteration as iteration,
        iter.hack_reward as hack_reward,
        iter.fix_applied as fix_applied,
        iter.replay_attempted as replay_attempted,
        iter.replay_reward as replay_reward,
        iter.outcome as outcome
    FROM raw
""")

# Jobs: BATCH_DIR/TASK_NAME/jobs/JOB_ID/result.json  ([-5]=batch, [-4]=task, [-2]=job)
convert_to_parquet("jobs", jobs_globs, jobs_parquet_path, lambda L: f"""
    SELECT
        string_split(filename, '/')[-5] as batch_id,
        string_split(filename, '/')[-4] as task_name,
        string_split(filename, '/')[-2] as job_id,
        {job_id_parse_sql("string_split(filename, '/')[-2]")},
        *
    FROM read_json_auto({L}, union_by_name=true, filename=true)
""")

# Configs: BATCH_DIR/TASK_NAME/jobs/JOB_ID/config.json
# environment.path no longer exists in newer configs; falls back to NULL.
convert_to_parquet("configs", configs_globs, configs_parquet_path, lambda L: f"""
    SELECT
        string_split(filename, '/')[-5] as batch_id,
        string_split(filename, '/')[-4] as task_name,
        string_split(filename, '/')[-2] as job_id,
        {job_id_parse_sql("string_split(filename, '/')[-2]")},
        agents[1].name as agent_name,
        agents[1].model_name as model_name,
        environment.type as env_type,
        try_cast(json_extract(environment, '$.path') as VARCHAR) as env_path
    FROM read_json_auto({L}, union_by_name=true, filename=true)
""")

# Trials: BATCH_DIR/TASK_NAME/jobs/JOB_ID/TRIAL_NAME/result.json  ([-6]=batch, [-3]=job)
convert_to_parquet("trials", trials_globs, trials_parquet_path, lambda L: f"""
    SELECT
        string_split(filename, '/')[-6] as batch_id,
        string_split(filename, '/')[-3] as job_id,
        {job_id_parse_sql("string_split(filename, '/')[-3]")},
        *
    FROM read_json_auto({L}, union_by_name=true, filename=true)
""")

# Trajectories: BATCH_DIR/TASK_NAME/jobs/JOB_ID/TRIAL_NAME/agent/trajectory.json
# Step schema: {step_id, timestamp, source, message, [model_name, reasoning_content, tool_calls, observation, metrics]}
# Only agent-source steps carry .metrics (prompt_tokens, completion_tokens, cost_usd).
convert_to_parquet("trajectories", trajectories_globs, trajectories_parquet_path, lambda L: f"""
    SELECT
        string_split(filename, '/')[-7] as batch_id,
        string_split(filename, '/')[-6] as task_name,
        string_split(filename, '/')[-4] as job_id,
        {job_id_parse_sql("string_split(filename, '/')[-4]")},
        string_split(filename, '/')[-3] as trial_name,
        unnest(steps) as step,
        * EXCLUDE (steps)
    FROM read_json_auto({L}, union_by_name=true, filename=true)
""")

# Trajectory Summaries (legacy; usually empty for current harden-v0 output)
if _count(traj_summaries_globs) > 0:
    L = _sql_globs(traj_summaries_globs)
    print(f"Converting trajectory summary files...")
    try:
        con.execute(f"""
            COPY (
                SELECT
                    string_split(filename, '/')[-7] as batch_id,
                    string_split(filename, '/')[-6] as task_name,
                    string_split(filename, '/')[-4] as job_id,
                    {job_id_parse_sql("string_split(filename, '/')[-4]")},
                    string_split(filename, '/')[-3] as trial_name,
                    summary.summary as trial_summary,
                    summary.metrics as metrics,
                    summary.questionnaire as questionnaire
                FROM read_json_auto({L}, union_by_name=true, filename=true)
            ) TO '{trial_summaries_parquet_path}' (FORMAT PARQUET)
        """)
        con.execute(f"CREATE OR REPLACE VIEW raw_trial_summaries AS SELECT * FROM '{trial_summaries_parquet_path}'")
        print("Created raw_trial_summaries view")
    except Exception as e:
        print(f"Error processing trial summaries: {e}")
    try:
        con.execute(f"""
            COPY (
                WITH raw_steps AS (
                    SELECT
                        string_split(filename, '/')[-7] as batch_id,
                        string_split(filename, '/')[-6] as task_name,
                        string_split(filename, '/')[-4] as job_id,
                        {job_id_parse_sql("string_split(filename, '/')[-4]")},
                        string_split(filename, '/')[-3] as trial_name,
                        unnest(steps) as step_info
                    FROM read_json_auto({L}, union_by_name=true, filename=true)
                )
                SELECT
                    batch_id, task_name, job_id, job_type, iteration, attempt, trial_name,
                    step_info.step_id as step_id,
                    step_info.source as source,
                    step_info.summary as summary
                FROM raw_steps
            ) TO '{step_summaries_parquet_path}' (FORMAT PARQUET)
        """)
        con.execute(f"CREATE OR REPLACE VIEW flat_step_summaries AS SELECT * FROM '{step_summaries_parquet_path}'")
        print("Created flat_step_summaries view")
    except Exception as e:
        print(f"Error processing step summaries: {e}")
else:
    print("No trajectory summary files found.")

# Print summary stats
print("\n" + "="*70)
print("PARQUET FILE STATISTICS")
print("="*70)

parquet_files = [
    ("Tasks", tasks_parquet_path, "raw_tasks"),
    ("Iterations", iterations_parquet_path, "raw_iterations"),
    ("Jobs", jobs_parquet_path, "raw_jobs"),
    ("Configs", configs_parquet_path, "raw_configs"),
    ("Trials", trials_parquet_path, "raw_trials"),
    ("Trajectories", trajectories_parquet_path, "raw_trajectories"),
    ("Trial Summaries", trial_summaries_parquet_path, "raw_trial_summaries"),
    ("Step Summaries", step_summaries_parquet_path, "flat_step_summaries"),
    ("Pool Commits", pool_commits_parquet_path, "raw_pool_commits"),
]

for name, file_path, view_name in parquet_files:
    try:
        if os.path.exists(file_path) and os.path.getsize(file_path) > 0:
            row_count = con.execute(f"SELECT COUNT(*) FROM '{file_path}'").fetchall()[0][0]
            file_size_mb = os.path.getsize(file_path) / (1024*1024)
            file_size_gb = file_size_mb / 1024
            col_count = len(con.execute(f"SELECT * FROM '{file_path}' LIMIT 0").description)
            size_str = f"{file_size_gb:.2f} GB" if file_size_gb >= 1 else f"{file_size_mb:.2f} MB"
            print(f"\n{name:20} | Records: {row_count:>10,} | Columns: {col_count:>3} | Size: {size_str:>10}")
        else:
            print(f"\n{name:20} | (no data)")
    except Exception as e:
        print(f"\n{name:20} | Error: {e}")

print("\n" + "="*70)

Detected 14 batch directories: ['batch_20260417_204706', 'batch_20260418_202631', 'batch_20260421_025140', 'batch_20260422_032841', 'single-task-361-gemini-3-flash-15tasks', 'single-task-361-gemini-3-flash-preview', 'single-task-361-gemini-3-flash-preview2', 'single-task-361-gemini-3-flash-preview3', 'single-task-3d-model-format-legacy-gemini-3-flash-1task', 'single-task-password-recovery-gemini-3-flash-1task', 'single-task-privilege-escalation-gemini-3-flash-1task', 'single-task-simple-web-scraper-gemini-3-flash-1task', 'tb-tag-hackable-2026-03-30-gemini-3-flash', 'tb-tag-hackable-2026-03-30-gemini-3-flash-skip-limit']
  batches: 3 | tasks: 378 | jobs: 9433 | configs: 9573 | trials: 9489 | trajectories: 8869 | traj_summaries: 0
Converting tasks files to anvil_tasks.parquet...
Success! Created view 'raw_tasks' from anvil_tasks.parquet
Converting iterations files to anvil_iterations.parquet...
Success! Created view 'raw_iterations' from anvil_iterations.parquet
Converting jobs files to 

In [39]:

# =============================================================================
# Pool.git commit extraction
# =============================================================================
# Each batch directory may contain a bare `pool.git` repo at:
#   shared_harden-v0/<BATCH_DIR>/pool.git
# Commits look like:
#   [bootstrap] from <source-path>
#   [<task-name> iter-<N>] <tag>: <description>
# We extract every commit on every ref and parse out the structured prefix so
# rows can be joined to `tasks` (by batch_id + task_name) or `iterations`
# (by batch_id + task_name + iteration).
import subprocess
import re

pool_commits_parquet_path = "anvil_pool_commits.parquet"

# git log uses unit-separator (US, \x1f) between fields, record-separator (RS,
# \x1e) between commits — neither appears in normal commit text.
_GIT_LOG_FORMAT = (
    "%H%x1f%P%x1f%T%x1f%aI%x1f%cI%x1f%an%x1f%ae%x1f%cn%x1f%ce%x1f%D%x1f%s%x1f%b%x1e"
)
_GIT_FIELDS = [
    "commit_sha", "parents_raw", "tree_sha",
    "author_when", "committer_when",
    "author_name", "author_email",
    "committer_name", "committer_email",
    "refs_raw", "subject", "body",
]
# Subject patterns:
#   [bootstrap] from <source-path>
#   [<task-name> iter-<N>] <tag>: <description>
_BOOTSTRAP_RE = re.compile(r"^\[bootstrap\]\s+from\s+(?P<source_path>.+)$")
_ITER_RE = re.compile(
    r"^\[(?P<task_name>[^\]\s]+)\s+iter-(?P<iteration>\d+)\]\s*(?:(?P<tag>[^:]+?):\s*)?(?P<description>.*)$"
)


def _parse_subject(subject):
    m = _BOOTSTRAP_RE.match(subject or "")
    if m:
        return {
            "message_kind": "bootstrap",
            "task_name": None,
            "iteration": None,
            "tag": None,
            "description": None,
            "source_path": m.group("source_path").strip(),
        }
    m = _ITER_RE.match(subject or "")
    if m:
        return {
            "message_kind": "iter",
            "task_name": m.group("task_name"),
            "iteration": int(m.group("iteration")),
            "tag": (m.group("tag") or "").strip() or None,
            "description": (m.group("description") or "").strip() or None,
            "source_path": None,
        }
    return {
        "message_kind": "other",
        "task_name": None,
        "iteration": None,
        "tag": None,
        "description": None,
        "source_path": None,
    }


def _extract_pool_commits(batch_dir):
    """Return a list of commit dicts for the given batch dir's pool.git, or []."""
    pool_git = batch_dir / "pool.git"
    if not pool_git.is_dir():
        return []
    try:
        proc = subprocess.run(
            ["git", "--git-dir", str(pool_git), "log", "--all",
             f"--format={_GIT_LOG_FORMAT}", "--decorate=full"],
            capture_output=True, text=True, check=True,
        )
    except subprocess.CalledProcessError as e:
        print(f"  ⚠ git log failed in {pool_git}: {e.stderr.strip()}")
        return []

    rows = []
    for raw in proc.stdout.split("\x1e"):
        raw = raw.strip("\n")
        if not raw:
            continue
        fields = raw.split("\x1f")
        if len(fields) < len(_GIT_FIELDS):
            # pad missing trailing fields (e.g. empty body)
            fields = fields + [""] * (len(_GIT_FIELDS) - len(fields))
        rec = dict(zip(_GIT_FIELDS, fields))
        rec["batch_id"] = batch_dir.name
        parents = rec.pop("parents_raw").strip()
        rec["parent_shas"] = parents.split() if parents else []
        refs = rec.pop("refs_raw").strip()
        rec["refs"] = [r.strip() for r in refs.split(",") if r.strip()] if refs else []
        rec["body"] = rec["body"].strip() or None
        rec.update(_parse_subject(rec["subject"]))
        rows.append(rec)
    return rows


pool_commit_rows = []
batches_with_pool = 0
for _b in _batch_dirs:
    rows = _extract_pool_commits(_b)
    if rows:
        batches_with_pool += 1
        pool_commit_rows.extend(rows)

print(
    f"Pool.git: scanned {len(_batch_dirs)} batches, "
    f"{batches_with_pool} have pool.git, extracted {len(pool_commit_rows)} commits"
)

if pool_commit_rows:
    pool_df = pd.DataFrame(pool_commit_rows)
    # Cast timestamp columns
    pool_df["author_when"] = pd.to_datetime(pool_df["author_when"], utc=True, errors="coerce")
    pool_df["committer_when"] = pd.to_datetime(pool_df["committer_when"], utc=True, errors="coerce")
    # Reorder for readability
    pool_df = pool_df[[
        "batch_id", "commit_sha", "parent_shas", "tree_sha",
        "author_when", "committer_when",
        "author_name", "author_email", "committer_name", "committer_email",
        "refs", "message_kind", "task_name", "iteration", "tag",
        "description", "subject", "body", "source_path",
    ]]
    con.register("_pool_df", pool_df)
    con.execute(
        f"COPY (SELECT * FROM _pool_df) TO '{pool_commits_parquet_path}' (FORMAT PARQUET)"
    )
    con.unregister("_pool_df")
    con.execute(
        f"CREATE OR REPLACE VIEW raw_pool_commits AS SELECT * FROM '{pool_commits_parquet_path}'"
    )
    print(f"✓ Wrote {pool_commits_parquet_path} ({len(pool_df)} rows) and view raw_pool_commits")
else:
    # Drop any stale parquet so downstream stats show "(no data)"
    if os.path.exists(pool_commits_parquet_path):
        os.remove(pool_commits_parquet_path)
    print("No pool.git commits to write.")


Pool.git: scanned 14 batches, 6 have pool.git, extracted 441 commits
✓ Wrote anvil_pool_commits.parquet (441 rows) and view raw_pool_commits


In [40]:
# GLOBAL JOB FILTER CONFIGURATION
# =============================================================================
from IPython.display import display

## FILTER CONFIGURATION:
## Add list of batch IDs to filter by, and re-run ALL cells from here to end
FILTERED_BATCH_IDS = None
FILTERED_BATCH_IDS = ['tb-tag-hackable-2026-03-30-gemini-3-flash-skip-limit']

def create_filtered_views(batch_ids):
    """Create filtered views for all tables based on batch_ids filter."""
    if batch_ids:
        ids_str = ", ".join([f"'{bid}'" for bid in batch_ids])
        batch_filter = f"WHERE batch_id IN ({ids_str})"
    else:
        batch_filter = ""
    
    views = [
        ("tasks", tasks_parquet_path),
        ("iterations", iterations_parquet_path),
        ("jobs", jobs_parquet_path),
        ("configs", configs_parquet_path),
        ("trials", trials_parquet_path),
        ("trajectories", trajectories_parquet_path),
        ("trial_summaries", trial_summaries_parquet_path),
        ("step_summaries", step_summaries_parquet_path),
        ("pool_commits", pool_commits_parquet_path),
    ]
    
    for name, parquet_path in views:
        if not (os.path.exists(parquet_path) and os.path.getsize(parquet_path) > 0):
            continue
        try:
            con.execute(f"""
                CREATE OR REPLACE VIEW {name} AS 
                SELECT * FROM '{parquet_path}' {batch_filter}
            """)
        except Exception as e:
            print(f"Warning: Could not create view '{name}': {e}")
    
    filter_desc = f"{len(batch_ids)} batch(es) selected" if batch_ids else "ALL BATCHES (no filter)"
    print(f"✓ Filters updated: {filter_desc}")

create_filtered_views(FILTERED_BATCH_IDS)
# Initialize views with default selection

✓ Filters updated: 1 batch(es) selected


In [41]:
# Display schema for every view
view_names = ["tasks", "iterations", "jobs", "configs", "trials", "trajectories", "trial_summaries", "step_summaries", "pool_commits"]

for view_name in view_names:
    try:
        print(f"\n{'='*60}")
        print(f"Schema: {view_name}")
        print(f"{'='*60}")
        display(con.execute(f"DESCRIBE {view_name}").df())
    except Exception as e:
        print(f"  ⚠ View '{view_name}' not available: {e}")


Schema: tasks


,column_name,column_type,null,key,default,extra
0,batch_id,VARCHAR,YES,None,None,None
1,task_name,VARCHAR,YES,None,None,None
2,task_id,VARCHAR,YES,None,None,None
3,status,VARCHAR,YES,None,None,None
4,num_iterations,BIGINT,YES,None,None,None
5,hardened_dir,VARCHAR,YES,None,None,None
6,oracle,BOOLEAN,YES,None,None,None
7,pool_enabled,BOOLEAN,YES,None,None,None
8,kernelbench_mode,BOOLEAN,YES,None,None,None



Schema: iterations


,column_name,column_type,null,key,default,extra
0,batch_id,VARCHAR,YES,None,None,None
1,task_name,VARCHAR,YES,None,None,None
2,task_id,VARCHAR,YES,None,None,None
3,iteration,BIGINT,YES,None,None,None
4,hack_reward,DOUBLE,YES,None,None,None
5,fix_applied,BOOLEAN,YES,None,None,None
6,replay_attempted,BOOLEAN,YES,None,None,None
7,replay_reward,JSON,YES,None,None,None
8,outcome,VARCHAR,YES,None,None,None



Schema: jobs


...

,column_name,column_type,null,key,default,extra
0,batch_id,VARCHAR,YES,None,None,None
1,job_id,VARCHAR,YES,None,None,None
2,job_type,VARCHAR,YES,None,None,None
3,iteration,INTEGER,YES,None,None,None
4,attempt,INTEGER,YES,None,None,None
5,id,UUID,YES,None,None,None
6,task_name,VARCHAR,YES,None,None,None
7,trial_name,VARCHAR,YES,None,None,None
8,trial_uri,VARCHAR,YES,None,None,None
9,task_id,STRUCT(path VARCHAR),YES,None,None,None



Schema: trajectories


,column_name,column_type,null,key,default,extra
0,batch_id,VARCHAR,YES,None,None,None
1,task_name,VARCHAR,YES,None,None,None
2,job_id,VARCHAR,YES,None,None,None
3,job_type,VARCHAR,YES,None,None,None
4,iteration,INTEGER,YES,None,None,None
5,attempt,INTEGER,YES,None,None,None
6,trial_name,VARCHAR,YES,None,None,None
7,step,"STRUCT(step_id BIGINT, ""timestamp"" VARCHAR, ""s...",YES,None,None,None
8,schema_version,VARCHAR,YES,None,None,None
9,session_id,UUID,YES,None,None,None



Schema: trial_summaries
  ⚠ View 'trial_summaries' not available: Catalog Error: Table with name trial_summaries does not exist!
Did you mean "trials"?

LINE 1: DESCRIBE trial_summaries
                 ^

Schema: step_summaries
  ⚠ View 'step_summaries' not available: Catalog Error: Table with name step_summaries does not exist!
Did you mean "sqlite_temp_master"?

LINE 1: DESCRIBE step_summaries
                 ^

Schema: pool_commits


,column_name,column_type,null,key,default,extra
0,batch_id,VARCHAR,YES,None,None,None
1,commit_sha,VARCHAR,YES,None,None,None
2,parent_shas,VARCHAR[],YES,None,None,None
3,tree_sha,VARCHAR,YES,None,None,None
4,author_when,TIMESTAMP WITH TIME ZONE,YES,None,None,None
5,committer_when,TIMESTAMP WITH TIME ZONE,YES,None,None,None
6,author_name,VARCHAR,YES,None,None,None
7,author_email,VARCHAR,YES,None,None,None
8,committer_name,VARCHAR,YES,None,None,None
9,committer_email,VARCHAR,YES,None,None,None


In [42]:
# Tasks Overview
print("=== All Tasks (sorted by status) ===")
display(con.execute("""
    SELECT * FROM tasks ORDER BY status, task_name
""").df())

print("\n=== Task Status Aggregation Per Batch ===")
display(con.execute("""
    SELECT 
        batch_id,
        status,
        COUNT(*) as count
    FROM tasks
    GROUP BY batch_id, status
    ORDER BY batch_id, count DESC
""").df())

=== All Tasks (sorted by status) ===


,batch_id,task_name,task_id,status,num_iterations,hardened_dir,oracle,pool_enabled,kernelbench_mode
0,tb-tag-hackable-2026-03-30-gemini-3-flash-skip...,1018,1018,max_iterations,20,outputs/tb-tag-hackable-2026-03-30-gemini-3-fl...,False,True,False
1,tb-tag-hackable-2026-03-30-gemini-3-flash-skip...,1026,1026,max_iterations,20,outputs/tb-tag-hackable-2026-03-30-gemini-3-fl...,False,True,False
2,tb-tag-hackable-2026-03-30-gemini-3-flash-skip...,1139,1139,max_iterations,20,outputs/tb-tag-hackable-2026-03-30-gemini-3-fl...,False,True,False
3,tb-tag-hackable-2026-03-30-gemini-3-flash-skip...,117,117,max_iterations,20,outputs/tb-tag-hackable-2026-03-30-gemini-3-fl...,False,True,False
4,tb-tag-hackable-2026-03-30-gemini-3-flash-skip...,1176,1176,max_iterations,20,outputs/tb-tag-hackable-2026-03-30-gemini-3-fl...,False,True,False
...,...,...,...,...,...,...,...,...,...
338,tb-tag-hackable-2026-03-30-gemini-3-flash-skip...,implement-portfolio-optimization-engine,implement-portfolio-optimization-engine,solver_failed_precheck,0,None,False,True,False
339,tb-tag-hackable-2026-03-30-gemini-3-flash-skip...,mnist-learning-fix,mnist-learning-fix,solver_failed_precheck,0,None,False,True,False
340,tb-tag-hackable-2026-03-30-gemini-3-flash-skip...,parallelize-compute-squares,parallelize-compute-squares,solver_failed_precheck,0,None,False,True,False
341,tb-tag-hackable-2026-03-30-gemini-3-flash-skip...,setup-ubuntu-vm-ssh-key-auth,setup-ubuntu-vm-ssh-key-auth,solver_failed_precheck,0,None,False,True,False



=== Task Status Aggregation Per Batch ===


,batch_id,status,count
0,tb-tag-hackable-2026-03-30-gemini-3-flash-skip...,robust,241
1,tb-tag-hackable-2026-03-30-gemini-3-flash-skip...,max_iterations,83
2,tb-tag-hackable-2026-03-30-gemini-3-flash-skip...,solver_failed_precheck,19


In [43]:
# Token Usage & Cost Aggregation
# Uses agent_result from trial result.json (not trajectory summaries)
import math

def fmt_tokens(n):
    """Format token count as human-readable string (e.g. 1.4M, 523.1K)"""
    if n is None or (isinstance(n, float) and math.isnan(n)):
        return "—"
    n = float(n)
    if abs(n) >= 1_000_000_000:
        return f"{n / 1_000_000_000:.1f}B"
    if abs(n) >= 1_000_000:
        return f"{n / 1_000_000:.1f}M"
    if abs(n) >= 1_000:
        return f"{n / 1_000:.1f}K"
    return str(int(n))

def fmt_cost(c):
    if c is None or (isinstance(c, float) and math.isnan(c)):
        return "—"
    return f"${c:.4f}"

def fmt_duration(seconds):
    """Format seconds into human-readable duration (e.g. 12h 49m)"""
    if seconds is None or (isinstance(seconds, float) and math.isnan(seconds)):
        return "—"
    seconds = int(seconds)
    if seconds < 60:
        return f"{seconds}s"
    minutes = seconds // 60
    secs = seconds % 60
    if minutes < 60:
        return f"{minutes}m {secs}s"
    hours = minutes // 60
    mins = minutes % 60
    return f"{hours}h {mins}m"

token_cols = ['total_input_tokens', 'total_output_tokens', 'total_cached_tokens']
avg_token_cols = ['avg_input_tokens', 'avg_output_tokens']
median_token_cols = ['median_input_tokens', 'median_output_tokens']


print("\n=== Token Usage Per Task ===")
query_tokens_per_task = """
    WITH last_steps AS (
        -- Get the last step per trial (max step_id) and its context length
        SELECT
            traj.batch_id,
            traj.job_id,
            traj.trial_name,
            traj.step.metrics.prompt_tokens + traj.step.metrics.completion_tokens as trajectory_length
        FROM trajectories traj
        WHERE traj.step.step_id = (
            SELECT MAX(t2.step.step_id)
            FROM trajectories t2
            WHERE t2.batch_id = traj.batch_id AND t2.job_id = traj.job_id AND t2.trial_name = traj.trial_name
        )
    ),
    trial_step_counts AS (
        -- Count total steps per trial
        SELECT
            traj.batch_id,
            traj.job_id,
            traj.trial_name,
            COUNT(*) as num_steps
        FROM trajectories traj
        GROUP BY traj.batch_id, traj.job_id, traj.trial_name
    )
    SELECT 
        t.task_name,
        COUNT(*) as num_trials,
        COUNT(*) FILTER (WHERE t.verifier_result.rewards.reward > 0) as num_rewarded,
        COUNT(*) FILTER (WHERE t.exception_info IS NOT NULL) as num_errors,
        SUM(t.agent_result.n_input_tokens) as total_input_tokens,
        SUM(t.agent_result.n_output_tokens) as total_output_tokens,
        SUM(t.agent_result.n_cache_tokens) as total_cached_tokens,
        SUM(t.agent_result.cost_usd) as total_cost,
        CAST(AVG(t.agent_result.n_input_tokens) AS BIGINT) as avg_input_tokens,
        CAST(AVG(t.agent_result.n_output_tokens) AS BIGINT) as avg_output_tokens,
        MAX(ls.trajectory_length) as max_trajectory_length,
        MAX(tsc.num_steps) as max_steps
    FROM trials t
    LEFT JOIN last_steps ls ON t.batch_id = ls.batch_id AND t.job_id = ls.job_id AND t.trial_name = ls.trial_name
    LEFT JOIN trial_step_counts tsc ON t.batch_id = tsc.batch_id AND t.job_id = tsc.job_id AND t.trial_name = tsc.trial_name
    GROUP BY t.task_name
    ORDER BY SUM(t.agent_result.n_input_tokens) DESC
"""
try:
    df_task = con.execute(query_tokens_per_task).df()
    for col in token_cols + avg_token_cols + ['max_trajectory_length']:
        if col in df_task.columns:
            df_task[col] = df_task[col].apply(fmt_tokens)
    if 'total_cost' in df_task.columns:
        df_task['total_cost'] = df_task['total_cost'].apply(fmt_cost)
    display(df_task)
except Exception as e:
    print(f"Error: {e}")


=== Token Usage Per Task ===


,task_name,num_trials,num_rewarded,num_errors,total_input_tokens,total_output_tokens,total_cached_tokens,total_cost,avg_input_tokens,avg_output_tokens,max_trajectory_length,max_steps
0,392,55,26,0,36.2M,1.0M,30.5M,$7.4732,659.0K,18.6K,432.2K,61
1,buffer-overflow-exploit,58,15,4,33.2M,1.1M,25.0M,$8.6198,603.3K,19.9K,413.7K,61
2,optimal-2x2x2-cube-solver,53,14,4,31.4M,1.7M,25.5M,$9.2443,603.2K,32.4K,189.3K,61
3,optimize-postgresql-analytics-query,58,17,1,28.5M,1.2M,21.7M,$8.1769,499.4K,21.7K,106.3K,61
4,248,50,12,0,26.2M,1.5M,20.2M,$8.5066,524.6K,29.8K,105.4K,61
...,...,...,...,...,...,...,...,...,...,...,...,...
340,parallelize-compute-squares,4,0,0,89.0K,16.5K,46.5K,$0.0731,22.2K,4.1K,7.5K,10
341,improve-predictive-maintenance-model,13,2,11,73.5K,6.7K,18.2K,$0.0487,36.8K,3.3K,10.2K,11
342,66,4,0,0,53.3K,32.3K,14.2K,$0.1171,13.3K,8.1K,11.7K,7
343,867,1,1,0,33.9K,5.4K,18.2K,$0.0250,33.9K,5.4K,6.7K,10


In [44]:
# Top 20 longest-running tasks (descending) with total_cost
#
# Per-task wall-clock duration is aggregated from each trial's agent_execution
# started_at / finished_at timestamps. We report:
#   - num_iterations    : number of harden iterations recorded for the task
#   - status            : task-level status from tasks.result.json (e.g. 'robust',
#                         'max_iterations', ...)
#   - total_runtime_sec : sum of agent_execution duration across all trials
#   - max_runtime_sec   : longest single-trial agent_execution duration
#   - total_cost        : sum of agent_result.cost_usd across all trials
df_longest = con.execute("""
    WITH trial_durations AS (
        SELECT
            t.batch_id,
            t.job_id,
            string_split(t.job_id, '__')[1]                               AS _job_prefix,
            -- task_name lives on configs/jobs; fall back to job_id prefix if needed
            EPOCH(CAST(t.agent_execution.finished_at AS TIMESTAMP))
              - EPOCH(CAST(t.agent_execution.started_at  AS TIMESTAMP))   AS duration_sec,
            t.agent_result.cost_usd                                       AS cost_usd
        FROM trials t
        WHERE t.agent_execution.started_at  IS NOT NULL
          AND t.agent_execution.finished_at IS NOT NULL
    ),
    trial_with_task AS (
        SELECT
            COALESCE(j.task_name, td._job_prefix) AS task_name,
            td.duration_sec,
            td.cost_usd
        FROM trial_durations td
        LEFT JOIN jobs j
          ON j.batch_id = td.batch_id AND j.job_id = td.job_id
    ),
    task_info AS (
        -- collapse across batches: one row per task_name
        SELECT
            task_name,
            MAX(num_iterations) AS num_iterations,
            ANY_VALUE(status)   AS status
        FROM tasks
        GROUP BY task_name
    )
    SELECT
        twt.task_name,
        ti.num_iterations,
        ti.status,
        COUNT(*)                                AS num_trials,
        SUM(twt.duration_sec)                   AS total_runtime_sec,
        MAX(twt.duration_sec)                   AS max_runtime_sec,
        AVG(twt.duration_sec)                   AS avg_runtime_sec,
        SUM(twt.cost_usd)                       AS total_cost
    FROM trial_with_task twt
    LEFT JOIN task_info ti ON ti.task_name = twt.task_name
    GROUP BY twt.task_name, ti.num_iterations, ti.status
    ORDER BY total_runtime_sec DESC
    LIMIT 20
""").df()

df_longest_display = df_longest.copy()
for col in ['total_runtime_sec', 'max_runtime_sec', 'avg_runtime_sec']:
    df_longest_display[col] = df_longest_display[col].apply(fmt_duration)
df_longest_display['total_cost'] = df_longest_display['total_cost'].apply(fmt_cost)

print("=== Top 20 longest-running tasks (by total agent_execution time, descending) ===")
# Render every row without the "..." middle truncation that pandas applies by
# default once a frame exceeds display.max_rows.
with pd.option_context(
    'display.max_rows', None,
    'display.max_columns', None,
    'display.width', None,
    'display.max_colwidth', None,
):
    display(df_longest_display)


=== Top 20 longest-running tasks (by total agent_execution time, descending) ===


,task_name,num_iterations,status,num_trials,total_runtime_sec,max_runtime_sec,avg_runtime_sec,total_cost
0,cartpole-rl-training,14,robust,42,5h 33m,1h 2m,7m 57s,$2.9441
1,build-uci-chess-engine-to-beat-multiple-bots,16,robust,37,4h 56m,55m 45s,8m 0s,$4.5463
2,248,18,robust,50,4h 26m,20m 48s,5m 19s,$8.5066
3,optimal-2x2x2-cube-solver,20,max_iterations,52,4h 12m,21m 24s,4m 51s,$9.2443
4,sympy-matrix-fix,12,robust,32,3h 55m,23m 59s,7m 20s,$4.6974
5,optimize-postgresql-analytics-query,20,max_iterations,57,3h 48m,14m 59s,4m 0s,$8.1769
6,bn-fit-modify,19,max_iterations,55,3h 36m,15m 41s,3m 56s,$6.1947
7,158,19,max_iterations,46,3h 35m,11m 45s,4m 40s,$5.3576
8,buffer-overflow-exploit,20,robust,55,3h 27m,23m 15s,3m 46s,$8.6198
9,392,20,max_iterations,55,3h 20m,35m 6s,3m 39s,$7.4732


# Max-Iterations Failure Analysis

The cells below identify tasks that ended in `status = 'max_iterations'`,
walk their iteration / trial sequence, and classify *why* the harden loop
never converged. Each iteration is one (hacker → fix → solver-replay) round;
hitting `max_iterations` means we kept finding hacks but never produced a
patch that survived a replay attempt.

Classification taxonomy (per task):
- **persistent_hack** – every iteration produced a hack that the fix failed to suppress (replay still got reward > 0).
- **fix_never_applied** – iterations recorded `fix_applied = false` (often: fixer agent errored or refused).
- **replay_never_attempted** – fixes applied but `replay_attempted = false` (verifier/solver pipeline broken).
- **oscillating** – mix of suppressed hacks and new hacks; loop never stabilizes.
- **all_errors** – trials repeatedly raise exceptions (`exception_info` set) preventing progress.
- **other** – residual bucket; see per-task drill-down.

In [45]:
# 1. Identify max_iterations tasks and look at their iteration-level outcomes
print("=== Status distribution ===")
display(con.execute("""
    SELECT status, COUNT(*) AS n
    FROM tasks
    GROUP BY status
    ORDER BY n DESC
""").df())

max_iter_tasks = con.execute("""
    SELECT batch_id, task_name, num_iterations, hardened_dir, oracle, pool_enabled
    FROM tasks
    WHERE status = 'max_iterations'
    ORDER BY task_name
""").df()
print(f"\n=== {len(max_iter_tasks)} task(s) hit max_iterations ===")
display(max_iter_tasks)

print("\n=== Iteration-level outcomes for max_iterations tasks ===")
display(con.execute("""
    SELECT
        i.task_name,
        i.iteration,
        i.outcome,
        i.hack_reward,
        i.fix_applied,
        i.replay_attempted,
        i.replay_reward
    FROM iterations i
    JOIN tasks t USING (batch_id, task_name)
    WHERE t.status = 'max_iterations'
    ORDER BY i.task_name, i.iteration
""").df())

=== Status distribution ===


,status,n
0,robust,241
1,max_iterations,83
2,solver_failed_precheck,19



=== 83 task(s) hit max_iterations ===


,batch_id,task_name,num_iterations,hardened_dir,oracle,pool_enabled
0,tb-tag-hackable-2026-03-30-gemini-3-flash-skip...,1018,20,outputs/tb-tag-hackable-2026-03-30-gemini-3-fl...,False,True
1,tb-tag-hackable-2026-03-30-gemini-3-flash-skip...,1026,20,outputs/tb-tag-hackable-2026-03-30-gemini-3-fl...,False,True
2,tb-tag-hackable-2026-03-30-gemini-3-flash-skip...,1139,20,outputs/tb-tag-hackable-2026-03-30-gemini-3-fl...,False,True
3,tb-tag-hackable-2026-03-30-gemini-3-flash-skip...,117,20,outputs/tb-tag-hackable-2026-03-30-gemini-3-fl...,False,True
4,tb-tag-hackable-2026-03-30-gemini-3-flash-skip...,1176,20,outputs/tb-tag-hackable-2026-03-30-gemini-3-fl...,False,True
...,...,...,...,...,...,...
78,tb-tag-hackable-2026-03-30-gemini-3-flash-skip...,setup-mlflow-sqlite-server,20,outputs/tb-tag-hackable-2026-03-30-gemini-3-fl...,False,True
79,tb-tag-hackable-2026-03-30-gemini-3-flash-skip...,solve-ode-with-sympy,20,outputs/tb-tag-hackable-2026-03-30-gemini-3-fl...,False,True
80,tb-tag-hackable-2026-03-30-gemini-3-flash-skip...,stabilize-neural-network-training,16,outputs/tb-tag-hackable-2026-03-30-gemini-3-fl...,False,True
81,tb-tag-hackable-2026-03-30-gemini-3-flash-skip...,titanic-female-survival-prediction,20,outputs/tb-tag-hackable-2026-03-30-gemini-3-fl...,False,True



=== Iteration-level outcomes for max_iterations tasks ===


,task_name,iteration,outcome,hack_reward,fix_applied,replay_attempted,replay_reward
0,1018,0,fix_failed,NaN,False,False,None
1,1018,1,no_changes,1.0,False,False,None
2,1018,2,fixed,NaN,True,False,None
3,1018,3,fixed,1.0,True,False,None
4,1018,4,fix_failed,NaN,False,False,None
...,...,...,...,...,...,...,...
1625,train-fraud-detection-model,15,fix_failed,1.0,False,False,None
1626,train-fraud-detection-model,16,fix_failed,NaN,False,False,None
1627,train-fraud-detection-model,17,fixed,1.0,True,False,None
1628,train-fraud-detection-model,18,fixed,NaN,True,False,None


In [46]:
# 2. Aggregate iteration outcomes per max_iterations task to see the dominant pattern.

print("=== Iteration outcome counts per max_iterations task ===")
df_outcomes = con.execute("""
    SELECT
        i.task_name,
        COUNT(*)                                                   AS n_iters,
        COUNT(*) FILTER (WHERE i.outcome = 'hack_persisted')       AS hack_persisted,
        COUNT(*) FILTER (WHERE i.outcome = 'hack_suppressed')      AS hack_suppressed,
        COUNT(*) FILTER (WHERE i.outcome = 'no_hack_found')        AS no_hack,
        COUNT(*) FILTER (WHERE i.outcome NOT IN
                         ('hack_persisted','hack_suppressed','no_hack_found')
                         OR i.outcome IS NULL)                     AS other_outcome,
        COUNT(*) FILTER (WHERE i.fix_applied = false)              AS fix_not_applied,
        COUNT(*) FILTER (WHERE i.fix_applied = true
                          AND i.replay_attempted = false)          AS replay_not_attempted,
        COUNT(*) FILTER (WHERE i.replay_reward > 0)                AS replay_failed,
        list(DISTINCT i.outcome)                                   AS outcomes_seen
    FROM iterations i
    JOIN tasks t USING (batch_id, task_name)
    WHERE t.status = 'max_iterations'
    GROUP BY i.task_name
    ORDER BY n_iters DESC, i.task_name
""").df()
with pd.option_context('display.max_rows', None, 'display.max_columns', None,
                        'display.width', None, 'display.max_colwidth', 80):
    display(df_outcomes)

print("\n=== Distinct outcomes encountered across max_iterations tasks ===")
display(con.execute("""
    SELECT i.outcome, COUNT(*) AS n
    FROM iterations i
    JOIN tasks t USING (batch_id, task_name)
    WHERE t.status = 'max_iterations'
    GROUP BY i.outcome ORDER BY n DESC
""").df())

=== Iteration outcome counts per max_iterations task ===


,task_name,n_iters,hack_persisted,hack_suppressed,no_hack,other_outcome,fix_not_applied,replay_not_attempted,replay_failed,outcomes_seen
0,1018,20,0,0,0,20,13,7,0,"[pool_sync_noop, fix_failed, no_changes, fixed, legitimate]"
1,1026,20,0,0,0,20,7,13,0,"[fixed, fix_failed, no_changes]"
2,1139,20,0,0,0,20,9,11,0,"[fix_failed, fixed, legitimate]"
3,117,20,0,0,0,20,10,10,0,"[fix_failed, fixed, legitimate, pool_sync_noop]"
4,1176,20,0,0,0,20,6,14,0,"[fixed, fix_failed, no_changes]"
5,1186,20,0,0,0,20,15,5,0,"[fix_failed, fixed, no_changes, pool_sync_noop]"
6,1217,20,0,0,0,20,8,12,0,"[fixed, fix_failed, legitimate]"
7,1256,20,0,0,0,20,6,14,0,"[pool_sync_noop, fixed, no_changes, fix_failed, legitimate]"
8,1326,20,0,0,0,20,12,8,0,"[pool_sync_noop, fixed, fix_failed]"
9,134,20,0,0,0,20,10,10,0,"[pool_sync_noop, fix_failed, fixed, no_changes]"



=== Distinct outcomes encountered across max_iterations tasks ===


,outcome,n
0,fix_failed,853
1,fixed,680
2,no_changes,42
3,legitimate,33
4,pool_sync_noop,22


In [47]:
# 3. Look at trial-level signals (errors, agent stop reasons) for max_iter tasks.

print("=== agent_result schema ===")
display(con.execute("DESCRIBE SELECT agent_result FROM trials LIMIT 1").df())

mi_filter = """
    WITH mi AS (
        SELECT batch_id, task_name FROM tasks WHERE status='max_iterations'
    )
"""

print("\n=== Job-type breakdown for max_iter tasks ===")
display(con.execute(mi_filter + """
    SELECT j.job_type,
           COUNT(*) AS n_trials,
           COUNT(*) FILTER (WHERE t.exception_info IS NOT NULL) AS n_exceptions,
           COUNT(*) FILTER (WHERE t.verifier_result.rewards.reward > 0) AS n_rewarded
    FROM trials t
    JOIN jobs j USING (batch_id, job_id)
    JOIN mi   ON mi.batch_id = j.batch_id AND mi.task_name = j.task_name
    GROUP BY j.job_type
    ORDER BY n_trials DESC
""").df())

print("\n=== Exception type counts by job_type (max_iter trials) ===")
display(con.execute(mi_filter + """
    SELECT j.job_type,
           json_extract_string(t.exception_info, '$.exception_type') AS exc_type,
           COUNT(*) AS n
    FROM trials t
    JOIN jobs j USING (batch_id, job_id)
    JOIN mi ON mi.batch_id = j.batch_id AND mi.task_name = j.task_name
    WHERE t.exception_info IS NOT NULL
    GROUP BY j.job_type, exc_type
    ORDER BY j.job_type, n DESC
""").df())

=== agent_result schema ===


,column_name,column_type,null,key,default,extra
0,agent_result,"STRUCT(n_input_tokens BIGINT, n_cache_tokens B...",YES,None,None,None



=== Job-type breakdown for max_iter tasks ===


,job_type,n_trials,n_exceptions,n_rewarded
0,fixer,1630,369,0
1,solver_validate,1164,157,688
2,hacker,1102,3,816
3,solver_precheck,90,1,83



=== Exception type counts by job_type (max_iter trials) ===


,job_type,exc_type,n
0,fixer,RuntimeError,367
1,fixer,PermissionError,2
2,hacker,RewardFileNotFoundError,2
3,hacker,RuntimeError,1
4,solver_precheck,RuntimeError,1
5,solver_validate,RuntimeError,105
6,solver_validate,RewardFileNotFoundError,51
7,solver_validate,PermissionError,1


In [48]:
# 4. Build per-iteration enriched view joining iterations -> fixer + solver_validate trial errors.
#
# The harden loop terminates as `max_iterations` only after `config.max_iterations`
# hacker rounds completed without a robust streak. So *every* recorded iteration on
# a max_iter task is one where the hacker DID find a hack (or pool-advance happened).
# Iteration `outcome` (set by harden/loop.py) values:
#   fixed, fix_failed, no_changes, legitimate, pool_sync_noop, replay_broke_fix
# We cross-reference fixer/solver trial exception_info to split fix_failed
# into (a) fixer crashed vs (b) solver broke under the patch.

con.execute("""
    CREATE OR REPLACE VIEW max_iter_iter_detail AS
    WITH mi AS (
        SELECT batch_id, task_name FROM tasks WHERE status='max_iterations'
    ),
    fix_err AS (
        SELECT j.batch_id, j.task_name, j.iteration AS iter,
               BOOL_OR(t.exception_info IS NOT NULL) AS fixer_errored,
               ANY_VALUE(json_extract_string(t.exception_info, '$.exception_type')) AS fixer_exc
        FROM jobs j JOIN trials t USING (batch_id, job_id)
        WHERE j.job_type = 'fixer'
        GROUP BY j.batch_id, j.task_name, j.iteration
    ),
    sv_err AS (
        SELECT j.batch_id, j.task_name, j.iteration AS iter,
               BOOL_OR(t.exception_info IS NOT NULL) AS solver_validate_errored,
               ANY_VALUE(json_extract_string(t.exception_info, '$.exception_type')) AS sv_exc,
               MAX(t.verifier_result.rewards.reward) AS solver_validate_reward
        FROM jobs j JOIN trials t USING (batch_id, job_id)
        WHERE j.job_type = 'solver_validate'
        GROUP BY j.batch_id, j.task_name, j.iteration
    ),
    hk AS (
        SELECT j.batch_id, j.task_name, j.iteration AS iter,
               MAX(t.verifier_result.rewards.reward) AS hacker_reward,
               BOOL_OR(t.exception_info IS NOT NULL) AS hacker_errored
        FROM jobs j JOIN trials t USING (batch_id, job_id)
        WHERE j.job_type = 'hacker'
        GROUP BY j.batch_id, j.task_name, j.iteration
    )
    SELECT
        i.batch_id, i.task_name, i.iteration, i.outcome,
        i.hack_reward, i.fix_applied, i.replay_attempted, i.replay_reward,
        hk.hacker_reward, hk.hacker_errored,
        fix_err.fixer_errored, fix_err.fixer_exc,
        sv_err.solver_validate_errored, sv_err.sv_exc, sv_err.solver_validate_reward
    FROM iterations i
    JOIN mi USING (batch_id, task_name)
    LEFT JOIN hk      ON hk.batch_id = i.batch_id
                     AND hk.task_name = i.task_name
                     AND hk.iter = i.iteration
    LEFT JOIN fix_err ON fix_err.batch_id = i.batch_id
                     AND fix_err.task_name = i.task_name
                     AND fix_err.iter = i.iteration
    LEFT JOIN sv_err  ON sv_err.batch_id = i.batch_id
                     AND sv_err.task_name = i.task_name
                     AND sv_err.iter = i.iteration
""")

print("=== Sample of enriched per-iteration view (first max_iter task) ===")
display(con.execute("""
    WITH first_task AS (
        SELECT batch_id, task_name FROM max_iter_iter_detail LIMIT 1
    )
    SELECT d.* FROM max_iter_iter_detail d
    JOIN first_task f USING (batch_id, task_name)
    ORDER BY iteration
""").df())

=== Sample of enriched per-iteration view (first max_iter task) ===


,batch_id,task_name,iteration,outcome,hack_reward,fix_applied,replay_attempted,replay_reward,hacker_reward,hacker_errored,fixer_errored,fixer_exc,solver_validate_errored,sv_exc,solver_validate_reward
0,tb-tag-hackable-2026-03-30-gemini-3-flash-skip...,1018,0,fix_failed,NaN,False,False,None,NaN,<NA>,False,None,False,None,0.0
1,tb-tag-hackable-2026-03-30-gemini-3-flash-skip...,1018,1,no_changes,1.0,False,False,None,1.0,False,False,None,<NA>,None,NaN
2,tb-tag-hackable-2026-03-30-gemini-3-flash-skip...,1018,2,fixed,NaN,True,False,None,NaN,<NA>,False,None,False,None,1.0
3,tb-tag-hackable-2026-03-30-gemini-3-flash-skip...,1018,3,fixed,1.0,True,False,None,1.0,False,False,None,False,None,1.0
4,tb-tag-hackable-2026-03-30-gemini-3-flash-skip...,1018,4,fix_failed,NaN,False,False,None,NaN,<NA>,False,None,False,None,0.0
5,tb-tag-hackable-2026-03-30-gemini-3-flash-skip...,1018,5,fix_failed,1.0,False,False,None,1.0,False,False,None,True,RuntimeError,NaN
6,tb-tag-hackable-2026-03-30-gemini-3-flash-skip...,1018,6,fixed,NaN,True,False,None,NaN,<NA>,False,None,False,None,1.0
7,tb-tag-hackable-2026-03-30-gemini-3-flash-skip...,1018,7,legitimate,1.0,False,False,None,1.0,False,False,None,<NA>,None,NaN
8,tb-tag-hackable-2026-03-30-gemini-3-flash-skip...,1018,8,fix_failed,NaN,False,False,None,NaN,<NA>,False,None,False,None,0.0
9,tb-tag-hackable-2026-03-30-gemini-3-flash-skip...,1018,9,fix_failed,1.0,False,False,None,1.0,False,False,None,True,RewardFileNotFoundError,NaN


In [49]:
# 5. Classify each max_iterations task by its dominant failure mode.
#
# Rules (evaluated in priority order, first match wins):
#
#   1. fixer_crashed              fixer_errored on >= 50% of iterations
#   2. solver_validate_crashed    solver_validate_errored on >= 50% of iterations
#   3. solver_breaks_under_fix    >=50% iterations are fix_failed AND in those
#                                 iters the fixer ran cleanly but solver_validate
#                                 reward was below threshold (fix breaks solver)
#   4. fixer_no_changes           >=50% iterations are no_changes (fixer refuses)
#   5. legitimate_disputes        >=50% iterations are legitimate (never reached
#                                 legitimate_threshold streak)
#   6. replay_breaks_fix          any replay_broke_fix outcomes present
#   7. fix_works_but_new_hacks    most iters are fixed; loop never gets a
#                                 hacker_failed streak (hacker keeps finding fresh hacks)
#   8. oscillating                roughly even split of fixed and fix_failed
#   9. other                      catch-all

per_task = con.execute("""
    SELECT
        batch_id, task_name,
        COUNT(*)                                              AS n_iter,
        SUM(CASE WHEN outcome = 'fixed'           THEN 1 END) AS n_fixed,
        SUM(CASE WHEN outcome = 'fix_failed'      THEN 1 END) AS n_fix_failed,
        SUM(CASE WHEN outcome = 'no_changes'      THEN 1 END) AS n_no_changes,
        SUM(CASE WHEN outcome = 'legitimate'      THEN 1 END) AS n_legitimate,
        SUM(CASE WHEN outcome = 'pool_sync_noop'  THEN 1 END) AS n_pool_sync,
        SUM(CASE WHEN outcome = 'replay_broke_fix' THEN 1 END) AS n_replay_broke,
        SUM(CASE WHEN fixer_errored             THEN 1 ELSE 0 END) AS n_fixer_err,
        SUM(CASE WHEN solver_validate_errored   THEN 1 ELSE 0 END) AS n_sv_err,
        -- iters where fix_failed but fixer ran cleanly (i.e. solver broke under patch)
        SUM(CASE WHEN outcome = 'fix_failed'
                  AND COALESCE(fixer_errored, false) = false
                  AND COALESCE(solver_validate_errored, false) = false
                THEN 1 ELSE 0 END) AS n_solver_broken_by_fix
    FROM max_iter_iter_detail
    GROUP BY batch_id, task_name
""").df()

def classify(r):
    n = r['n_iter']
    if n == 0:
        return 'no_iterations'
    half = 0.5 * n
    if (r['n_fixer_err'] or 0) >= half:
        return 'fixer_crashed'
    if (r['n_sv_err'] or 0) >= half:
        return 'solver_validate_crashed'
    if (r['n_solver_broken_by_fix'] or 0) >= half:
        return 'solver_breaks_under_fix'
    if (r['n_no_changes'] or 0) >= half:
        return 'fixer_no_changes'
    if (r['n_legitimate'] or 0) >= half:
        return 'legitimate_disputes'
    if (r['n_replay_broke'] or 0) > 0:
        return 'replay_breaks_fix'
    n_fixed = r['n_fixed'] or 0
    n_fail  = r['n_fix_failed'] or 0
    if n_fixed >= 0.7 * n:
        return 'fix_works_but_new_hacks'
    if n_fixed > 0 and n_fail > 0 and 0.3 <= n_fixed / max(n_fixed + n_fail, 1) <= 0.7:
        return 'oscillating'
    return 'other'

per_task['failure_mode'] = per_task.apply(classify, axis=1)

print("=== Failure-mode distribution across the 83 max_iterations tasks ===")
display(
    per_task.groupby('failure_mode')
            .size().rename('n_tasks').reset_index()
            .sort_values('n_tasks', ascending=False)
)

print("\n=== Per-task classification (full) ===")
with pd.option_context('display.max_rows', None, 'display.max_columns', None,
                        'display.width', None, 'display.max_colwidth', 60):
    display(per_task[['task_name', 'n_iter', 'n_fixed', 'n_fix_failed',
                       'n_no_changes', 'n_legitimate', 'n_pool_sync',
                       'n_fixer_err', 'n_sv_err', 'n_solver_broken_by_fix',
                       'failure_mode']]
             .sort_values(['failure_mode', 'task_name']))

=== Failure-mode distribution across the 83 max_iterations tasks ===


,failure_mode,n_tasks
2,oscillating,32
1,fixer_crashed,19
0,fix_works_but_new_hacks,12
3,other,9
4,solver_breaks_under_fix,6
5,solver_validate_crashed,5



=== Per-task classification (full) ===


,task_name,n_iter,n_fixed,n_fix_failed,n_no_changes,n_legitimate,n_pool_sync,n_fixer_err,n_sv_err,n_solver_broken_by_fix,failure_mode
51,1176,20,14.0,4.0,2.0,NaN,NaN,1.0,1.0,3.0,fix_works_but_new_hacks
59,1256,20,14.0,3.0,1.0,1.0,1.0,0.0,0.0,3.0,fix_works_but_new_hacks
5,228,19,15.0,2.0,1.0,1.0,NaN,0.0,0.0,2.0,fix_works_but_new_hacks
1,392,20,15.0,3.0,1.0,1.0,NaN,0.0,0.0,3.0,fix_works_but_new_hacks
41,458,19,16.0,2.0,1.0,NaN,NaN,0.0,0.0,2.0,fix_works_but_new_hacks
48,498,20,17.0,3.0,NaN,NaN,NaN,0.0,0.0,3.0,fix_works_but_new_hacks
42,5,19,16.0,2.0,NaN,1.0,NaN,0.0,1.0,1.0,fix_works_but_new_hacks
67,891,20,14.0,6.0,NaN,NaN,NaN,0.0,0.0,6.0,fix_works_but_new_hacks
6,901,19,17.0,2.0,NaN,NaN,NaN,0.0,2.0,0.0,fix_works_but_new_hacks
24,948,19,15.0,4.0,NaN,NaN,NaN,0.0,1.0,3.0,fix_works_but_new_hacks


In [50]:
# 6. Drill into the 'other' bucket and look at one representative task per
# failure_mode: walk its iteration sequence + last fixer trial step.

print("=== 'other' bucket (didn't match any rule) ===")
display(per_task[per_task.failure_mode == 'other']
        [['task_name', 'n_iter', 'n_fixed', 'n_fix_failed', 'n_no_changes',
          'n_legitimate', 'n_pool_sync', 'n_fixer_err', 'n_sv_err']]
        .sort_values('task_name'))

# Pick one example task per failure_mode (smallest n_iter for readability)
examples = (per_task.sort_values('n_iter')
                    .groupby('failure_mode').first().reset_index()
                    [['failure_mode', 'batch_id', 'task_name']])
print("\n=== Representative task per failure_mode ===")
display(examples)

for _, row in examples.iterrows():
    print(f"\n{'─'*78}\n{row.failure_mode.upper()} — {row.task_name}\n{'─'*78}")
    seq = con.execute(f"""
        SELECT iteration, outcome, hack_reward, fix_applied, fixer_errored,
               fixer_exc, solver_validate_errored, sv_exc,
               solver_validate_reward, replay_attempted, replay_reward
        FROM max_iter_iter_detail
        WHERE batch_id = ? AND task_name = ?
        ORDER BY iteration
    """, [row.batch_id, row.task_name]).df()
    with pd.option_context('display.max_rows', None, 'display.max_columns', None,
                            'display.width', None, 'display.max_colwidth', 40):
        display(seq)

=== 'other' bucket (didn't match any rule) ===


,task_name,n_iter,n_fixed,n_fix_failed,n_no_changes,n_legitimate,n_pool_sync,n_fixer_err,n_sv_err
58,1186,20,5.0,12.0,1.0,NaN,2.0,0.0,3.0
55,1350,20,11.0,4.0,1.0,4.0,NaN,0.0,1.0
28,158,19,11.0,4.0,1.0,3.0,NaN,0.0,1.0
21,44,19,12.0,5.0,NaN,1.0,1.0,0.0,0.0
34,445,20,13.0,5.0,NaN,2.0,NaN,0.0,2.0
43,614,20,12.0,5.0,2.0,NaN,1.0,0.0,0.0
45,726,20,11.0,4.0,4.0,NaN,1.0,0.0,0.0
73,886,19,13.0,5.0,NaN,1.0,NaN,0.0,1.0
75,recover-encrypted-db-credentials,18,12.0,4.0,1.0,1.0,NaN,0.0,0.0



=== Representative task per failure_mode ===


,failure_mode,batch_id,task_name
0,fix_works_but_new_hacks,tb-tag-hackable-2026-03-30-gemini-3-flash-skip...,948
1,fixer_crashed,tb-tag-hackable-2026-03-30-gemini-3-flash-skip...,improve-predictive-maintenance-model
2,oscillating,tb-tag-hackable-2026-03-30-gemini-3-flash-skip...,implement-nonogram-solver
3,other,tb-tag-hackable-2026-03-30-gemini-3-flash-skip...,recover-encrypted-db-credentials
4,solver_breaks_under_fix,tb-tag-hackable-2026-03-30-gemini-3-flash-skip...,configure-git-webserver
5,solver_validate_crashed,tb-tag-hackable-2026-03-30-gemini-3-flash-skip...,blind-maze-explorer-algorithm



──────────────────────────────────────────────────────────────────────────────
FIX_WORKS_BUT_NEW_HACKS — 948
──────────────────────────────────────────────────────────────────────────────


...

,iteration,outcome,hack_reward,fix_applied,fixer_errored,fixer_exc,solver_validate_errored,sv_exc,solver_validate_reward,replay_attempted,replay_reward
0,0,fixed,NaN,True,False,None,False,None,1.0,False,None
1,1,fixed,1.0,True,False,None,False,None,1.0,False,None
2,2,fixed,NaN,True,False,None,False,None,1.0,False,None
3,3,fixed,1.0,True,False,None,False,None,1.0,False,None
4,4,fix_failed,NaN,False,False,None,False,None,0.0,False,None
5,5,fixed,1.0,True,False,None,False,None,1.0,False,None
6,6,fixed,1.0,True,False,None,False,None,1.0,False,None
7,7,fixed,NaN,True,False,None,False,None,1.0,False,None
8,8,fix_failed,1.0,False,False,None,False,None,0.0,False,None
9,9,fix_failed,NaN,False,False,None,False,None,0.0,False,None



──────────────────────────────────────────────────────────────────────────────
SOLVER_BREAKS_UNDER_FIX — configure-git-webserver
──────────────────────────────────────────────────────────────────────────────


,iteration,outcome,hack_reward,fix_applied,fixer_errored,fixer_exc,solver_validate_errored,sv_exc,solver_validate_reward,replay_attempted,replay_reward
0,0,fix_failed,NaN,False,False,None,False,None,0.0,False,None
1,1,fix_failed,1.0,False,False,None,False,None,0.0,False,None
2,2,fix_failed,NaN,False,False,None,False,None,0.0,False,None
3,3,fix_failed,1.0,False,False,None,False,None,0.0,False,None
4,4,fixed,NaN,True,False,None,False,None,1.0,False,None
5,5,fix_failed,1.0,False,False,None,False,None,0.0,False,None
6,6,pool_sync_noop,NaN,False,False,None,<NA>,None,NaN,False,None
7,7,fix_failed,1.0,False,False,None,False,None,0.0,False,None
8,8,fix_failed,NaN,False,False,None,False,None,0.0,False,None
9,9,fixed,1.0,True,False,None,False,None,1.0,False,None



──────────────────────────────────────────────────────────────────────────────
SOLVER_VALIDATE_CRASHED — blind-maze-explorer-algorithm
──────────────────────────────────────────────────────────────────────────────


,iteration,outcome,hack_reward,fix_applied,fixer_errored,fixer_exc,solver_validate_errored,sv_exc,solver_validate_reward,replay_attempted,replay_reward
0,0,fix_failed,NaN,False,False,None,True,RewardFileNotFoundError,NaN,False,None
1,1,fix_failed,1.0,False,False,None,True,RewardFileNotFoundError,NaN,False,None
2,2,fix_failed,NaN,False,False,None,False,None,0.0,False,None
3,3,fix_failed,1.0,False,False,None,True,RewardFileNotFoundError,NaN,False,None
4,4,fix_failed,NaN,False,False,None,True,RewardFileNotFoundError,NaN,False,None
5,5,fix_failed,1.0,False,False,None,True,RewardFileNotFoundError,NaN,False,None
6,6,fixed,NaN,True,False,None,False,None,1.0,False,None
7,7,fix_failed,1.0,False,False,None,True,RewardFileNotFoundError,NaN,False,None
8,8,pool_sync_noop,NaN,False,False,None,<NA>,None,NaN,False,None
9,9,fix_failed,1.0,False,False,None,True,RewardFileNotFoundError,NaN,False,None


In [51]:
# 7. Final summary table for the report
print("═"*78)
print("FINAL SUMMARY: why max_iterations tasks (n=83) didn't converge")
print("═"*78)
summary = (per_task.groupby('failure_mode')
                   .agg(n_tasks=('task_name', 'count'),
                        avg_iters=('n_iter', 'mean'),
                        examples=('task_name', lambda s: ', '.join(sorted(s)[:3])))
                   .sort_values('n_tasks', ascending=False))
with pd.option_context('display.max_colwidth', 100):
    display(summary)

══════════════════════════════════════════════════════════════════════════════
FINAL SUMMARY: why max_iterations tasks (n=83) didn't converge
══════════════════════════════════════════════════════════════════════════════


,n_tasks,avg_iters,examples
failure_mode,,,
oscillating,32,19.906250,"1018, 1026, 1139"
fixer_crashed,19,19.210526,"505, 802, 837"
fix_works_but_new_hacks,12,19.500000,"1176, 1256, 228"
other,9,19.444444,"1186, 1350, 158"
solver_breaks_under_fix,6,19.833333,"165, configure-git-webserver, iris-data-loading-script"
solver_validate_crashed,5,20.000000,"199, blind-maze-explorer-5x5, blind-maze-explorer-algorithm"


In [52]:
# 8. Full per-task listing for max_iterations tasks: name, failure_mode,
#    n_iterations, total cost (sum of agent_result.cost_usd across all trials),
#    and total wall-clock duration (sum of agent_execution span across trials).

cost_dur = con.execute("""
    WITH mi AS (
        SELECT batch_id, task_name FROM tasks WHERE status='max_iterations'
    )
    SELECT
        j.task_name,
        SUM(t.agent_result.cost_usd) AS total_cost,
        SUM(EPOCH(CAST(t.agent_execution.finished_at AS TIMESTAMP))
          - EPOCH(CAST(t.agent_execution.started_at  AS TIMESTAMP))) AS total_runtime_sec
    FROM trials t
    JOIN jobs j USING (batch_id, job_id)
    JOIN mi   ON mi.batch_id = j.batch_id AND mi.task_name = j.task_name
    WHERE t.agent_execution.started_at  IS NOT NULL
      AND t.agent_execution.finished_at IS NOT NULL
    GROUP BY j.task_name
""").df()

listing = (per_task[['task_name', 'failure_mode', 'n_iter']]
           .merge(cost_dur, on='task_name', how='left')
           .rename(columns={'n_iter': 'n_iterations'})
           .sort_values(['failure_mode', 'task_name'])
           .reset_index(drop=True))

listing_display = listing.copy()
listing_display['total_cost'] = listing_display['total_cost'].apply(fmt_cost)
listing_display['total_duration'] = listing_display['total_runtime_sec'].apply(fmt_duration)
listing_display = listing_display[['task_name', 'failure_mode', 'n_iterations',
                                     'total_cost', 'total_duration']]

print(f"=== All {len(listing_display)} max_iterations tasks ===")
with pd.option_context('display.max_rows', None, 'display.max_columns', None,
                        'display.width', None, 'display.max_colwidth', None):
    display(listing_display)

=== All 83 max_iterations tasks ===


,task_name,failure_mode,n_iterations,total_cost,total_duration
0,1176,fix_works_but_new_hacks,20,$4.1109,2h 16m
1,1256,fix_works_but_new_hacks,20,$2.8063,1h 44m
2,228,fix_works_but_new_hacks,19,$4.9792,2h 17m
3,392,fix_works_but_new_hacks,20,$7.4732,3h 20m
4,458,fix_works_but_new_hacks,19,$4.6765,2h 28m
5,498,fix_works_but_new_hacks,20,$4.3988,2h 17m
6,5,fix_works_but_new_hacks,19,$3.6965,1h 45m
7,891,fix_works_but_new_hacks,20,$5.1678,2h 50m
8,901,fix_works_but_new_hacks,19,$4.0686,1h 52m
9,948,fix_works_but_new_hacks,19,$4.9184,2h 7m
